# Property Data Cleaning Notebook
This notebook cleans the raw property data by applying a series of logical rules. All changes are accumulated in `df_updateded`, which is saved once at the end as `property_details_updated.csv`.

## 0. Import libraries

In [1]:
import pandas as pd
import numpy as np

## 1. Load the data
Read the CSV without converting "N/A" to NaN – we keep them as strings.

In [2]:
df_original = pd.read_csv('property_details.csv', dtype=str, keep_default_na=False)
df_updateded = df_original.copy()   # work on a copy to preserve original if needed
print(f"Original rows: {len(df_updateded)}")

Original rows: 30818


## 2. Remove rows where all data columns are "N/A"
We identify columns that hold actual property data (everything except `Link` and `Date_scraped`).
Rows where **all** of these columns are exactly "N/A" are dropped from the working dataframe.

In [3]:
data_columns = [col for col in df_updateded.columns if col not in ['Link', 'Date_scraped']]
mask_all_na = (df_updateded[data_columns] == 'N/A').all(axis=1)
df_updateded = df_updateded[~mask_all_na].reset_index(drop=True)
removed_count = mask_all_na.sum()
print(f"Rows with all data N/A removed: {removed_count}")

Rows with all data N/A removed: 11


## 3. Remove rows where Price is "N/A"
Properties without a price are not useful, so we drop them.

In [4]:
mask_price_na = df_updateded['Price'] == 'N/A'
df_updateded = df_updateded[~mask_price_na].reset_index(drop=True)
removed_price = mask_price_na.sum()
print(f"Rows with Price = N/A removed: {removed_price}")

Rows with Price = N/A removed: 470


## 4. Apply logical cleaning rules
These rules correct inconsistencies and fill missing values based on property characteristics.

### 4.1 Studios (subtype '05') – Furnished and Kitchen
Most studios are furnished and have a fully equipped kitchen. If those fields are 'N/A', we set them to '1'.

In [5]:
cond_furnished_studio = (df_updateded['Subtype of property'] == '05') & (df_updateded['Furnished'] == 'N/A')
df_updateded.loc[cond_furnished_studio, 'Furnished'] = '1'
print(f"Furnished (studio) → '1': {cond_furnished_studio.sum()} rows updated")

Furnished (studio) → '1': 169 rows updated


In [6]:
cond_kitchen_studio = (df_updateded['Subtype of property'] == '05') & (df_updateded['Fully equipped kitchen'] == 'N/A')
df_updateded.loc[cond_kitchen_studio, 'Fully equipped kitchen'] = '1'
print(f"Fully equipped kitchen (studio) → '1': {cond_kitchen_studio.sum()} rows updated")

Fully equipped kitchen (studio) → '1': 431 rows updated


### 4.2 Terrace and Garden – area consistency
If the indicator (`Terrace` / `Garden`) is '0', the corresponding area should also be '0'.

In [7]:
cond_terrace = (df_updateded['Terrace'] == '0') & (df_updateded['Surface terrace'] != '0')
df_updateded.loc[cond_terrace, 'Surface terrace'] = '0'
print(f"Terrace surface → '0' (indicator=0): {cond_terrace.sum()} rows updated")

cond_garden = (df_updateded['Garden'] == '0') & (df_updateded['Garden area'] != '0')
df_updateded.loc[cond_garden, 'Garden area'] = '0'
print(f"Garden area → '0' (indicator=0): {cond_garden.sum()} rows updated")

Terrace surface → '0' (indicator=0): 6763 rows updated
Garden area → '0' (indicator=0): 11696 rows updated


### 4.3 Swimming pool – mixed‑buildings, apartments and studios
- Mixed‑use buildings rarely have a private pool → set to '0' if unknown.
- Apartments ('01') and studios ('05') typically do not have a private pool → set to '0' if unknown.

In [8]:
cond_mixed_pool = (df_updateded['Link'].str.contains('mixed-building', na=False)) & (df_updateded['Swimming pool'] == 'N/A')
df_updateded.loc[cond_mixed_pool, 'Swimming pool'] = '0'
print(f"Mixed‑building pool → '0': {cond_mixed_pool.sum()} rows updated")

cond_apt_studio_pool = (df_updateded['Subtype of property'].isin(['01', '05'])) & (df_updateded['Swimming pool'] == 'N/A')
df_updateded.loc[cond_apt_studio_pool, 'Swimming pool'] = '0'
print(f"Apartment/studio pool → '0': {cond_apt_studio_pool.sum()} rows updated")

Mixed‑building pool → '0': 347 rows updated
Apartment/studio pool → '0': 8449 rows updated


### 4.4 Derived binary features
- If a property is furnished (`Furnished == '1'`) but kitchen info is missing, assume it has a fully equipped kitchen.
- For apartments and studios, any missing binary feature (`Garden`, `Terrace`, `Fireplace`, `Swimming pool`, `Garage`) is set to '0'.

In [9]:
cond_furn_kitchen = (df_updateded['Furnished'] == '1') & (df_updateded['Fully equipped kitchen'] == 'N/A')
df_updateded.loc[cond_furn_kitchen, 'Fully equipped kitchen'] = '1'
print(f"Furnished → Fully equipped kitchen: {cond_furn_kitchen.sum()} rows updated")

apartment_studio = df_updateded['Subtype of property'].isin(['01', '05'])
binary_cols = ['Garden', 'Terrace', 'Fireplace', 'Swimming pool', 'Garage']
for col in binary_cols:
    cond = apartment_studio & (df_updateded[col] == 'N/A')
    df_updateded.loc[cond, col] = '0'
    print(f"{col} (apartment/studio) → '0': {cond.sum()} rows updated")

Furnished → Fully equipped kitchen: 1031 rows updated
Garden (apartment/studio) → '0': 1534 rows updated
Terrace (apartment/studio) → '0': 434 rows updated
Fireplace (apartment/studio) → '0': 10996 rows updated
Swimming pool (apartment/studio) → '0': 0 rows updated
Garage (apartment/studio) → '0': 7276 rows updated


## 5. adding price per square meter column

In [12]:
# Convert to numeric, coercing errors to NaN
price_num = pd.to_numeric(df_updateded['Price'], errors='coerce')
surface_num = pd.to_numeric(df_updateded['Livable surface'], errors='coerce')

# Calculate price per m² (NaN will appear where either conversion failed)
df_updateded['price_per_m2'] = (price_num / surface_num).round(2)

# (Optional) Replace NaN with "N/A" if you need the string representation
df_updateded['price_per_m2'] = df_updateded['price_per_m2'].where(
    price_num.notna() & surface_num.notna(), 
    "N/A"
)

## 6. Save the final cleaned data
All changes have been applied to `df_updateded`. We now save it to a single CSV file.

In [13]:
df_updateded.to_csv('property_details_updated.csv', index=False)
print(f"Final cleaned data saved to 'property_details_updated.csv' ({len(df_updateded)} rows)")

Final cleaned data saved to 'property_details_updated.csv' (30337 rows)
